### Final Modeling 

In [1]:
import pandas as pd
import numpy as np

from linearmodels import PanelOLS

In [2]:
# this data has already been cleaned from a different file, see "Exported_Cleaned_Data.ipynb" and "data_cleaning.py" for further details
# this data contains poverty values for baseline year 2019, which is necessary for modeling
grade_data = pd.read_csv('cleaned_school_data_updated.csv')

In [3]:
# fix poverty values
grade_data['poverty_percentage'] = (grade_data['% Poverty'] * 100).round(3)

#### Individual Grade Data

In [4]:
# grade splits

data3 = grade_data[
    (grade_data['Grade'] == '3') & 
    (grade_data['Student Category'] == 'All Students') 
    # (grade_data['Report Category'] == 'School')
]


data4 = grade_data[
    (grade_data['Grade'] == '4') &  
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data5 = grade_data[
    (grade_data['Grade'] == '5') &
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data6 = grade_data[
    (grade_data['Grade'] == '6') &   
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data7 = grade_data[
    (grade_data['Grade'] == '7') &
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data8 = grade_data[
    (grade_data['Grade'] == '8') &
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]



#### Model For Each Grade - Mean Scale Score

In [12]:
import pandas as pd
from linearmodels import PanelOLS

def run_grade_model(data):

    grade_data = data
    
    # 2019 poverty rate as baseline for each school
    poverty_baseline = grade_data[grade_data['Year'] == 2019][
        ['DBN', 'poverty_percentage']
    ].rename(columns={'poverty_percentage': 'poverty_baseline'})
    
    # merge the baseline poverty rate back into the main data
    grade_data = grade_data.merge(poverty_baseline, on='DBN', how='left')
    
    # interaction terms using new poverty baseline(2019) for pre and post periods
    grade_data['poverty_pre'] = (
        grade_data['poverty_baseline'] * (grade_data['Year'] == 2018).astype(int)
    )
    grade_data['poverty_post'] = (
        grade_data['poverty_baseline'] * (grade_data['Year'] == 2022).astype(int)
    )
    
    # index for PanelOLS
    grade_data = grade_data.set_index(['DBN', 'Year'])
    
    # PanelOLS model with clustered standard errors at the school level,
    # controlling for school and year fixed effects, 
    # and weighted by number of students tested
    model = PanelOLS(
        dependent=grade_data['mean_scale_score'],
        exog=grade_data[['poverty_pre', 'poverty_post']],
        entity_effects=True,
        time_effects=True,
        weights=grade_data['number_tested']
    ).fit(cov_type='clustered', cluster_entity=True)
    
    return model



In [13]:
model_3 = run_grade_model(data3)

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [14]:
model_3

Dep. Variable:,mean_scale_score,R-squared:,0.0212
Estimator:,PanelOLS,R-squared (Between):,-0.0051
No. Observations:,2306,R-squared (Within):,-0.0484
Date:,"Wed, Apr 29 2026",R-squared (Overall):,-0.0051
Time:,16:05:52,Log-likelihood,-5627.8
Cov. Estimator:,Clustered,,
,,F-statistic:,16.581
Entities:,774,P-value,0.0000
Avg Obs:,2.9793,Distribution:,"F(2,1528)"
Min Obs:,2.0000,,
Max Obs:,3.0000,F-statistic (robust):,10.888


#### Model for Comprehension Levels - Grades 3 & 4

In [16]:
import pandas as pd
from linearmodels import PanelOLS

def run_grade_model_level1(data):

    grade_data = data
    
    # 2019 poverty rate as baseline for each school
    poverty_baseline = grade_data[grade_data['Year'] == 2019][
        ['DBN', 'poverty_percentage']
    ].rename(columns={'poverty_percentage': 'poverty_baseline'})
    
    # merge the baseline poverty rate back into the main data
    grade_data = grade_data.merge(poverty_baseline, on='DBN', how='left')
    
    # interaction terms using new poverty baseline(2019) for pre and post periods
    grade_data['poverty_pre'] = (
        grade_data['poverty_baseline'] * (grade_data['Year'] == 2018).astype(int)
    )
    grade_data['poverty_post'] = (
        grade_data['poverty_baseline'] * (grade_data['Year'] == 2022).astype(int)
    )
    
    # index for PanelOLS
    grade_data = grade_data.set_index(['DBN', 'Year'])
    
    # PanelOLS model with clustered standard errors at the school level,
    # controlling for school and year fixed effects, 
    # and weighted by number of students tested
    model = PanelOLS(
        dependent=grade_data['level_1_percentage'],
        exog=grade_data[['poverty_pre', 'poverty_post']],
        entity_effects=True,
        time_effects=True,
        weights=grade_data['number_tested']
    ).fit(cov_type='clustered', cluster_entity=True)
    
    return model




In [17]:
model_3_level1 = run_grade_model_level1(data3)

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [19]:
model_3_level1.summary

Dep. Variable:,level_1_percentage,R-squared:,0.0431
Estimator:,PanelOLS,R-squared (Between):,0.2867
No. Observations:,2306,R-squared (Within):,0.0512
Date:,"Wed, Apr 29 2026",R-squared (Overall):,0.2729
Time:,16:08:11,Log-likelihood,-6754.3
Cov. Estimator:,Clustered,,
,,F-statistic:,34.401
Entities:,774,P-value,0.0000
Avg Obs:,2.9793,Distribution:,"F(2,1528)"
Min Obs:,2.0000,,
Max Obs:,3.0000,F-statistic (robust):,34.355
